# 课后练习解答（05.06_long_dialogue_test）

本解答对应章节课后练习，共 15 题。

### 问题1（单选题）

**题目：** 构造多轮 prompt 时，历史轮次的正确顺序是？
A. system → 第1轮 user/assistant → ... → 最新 user
B. 最新 user 放最前
C. 随机排列
D. 只保留 assistant

**解答：** A

**解析：** 模型按从左到右读取上下文，历史必须保持时间顺序，最新问题放在末尾。


### 问题2（单选题）

**题目：** tokenizer 默认 truncation_side="right"，对长对话的含义是？
A. 从文本末尾截断，可能丢掉最新 user 输入
B. 从开头截断
C. 不影响
D. 只截断 assistant

**解答：** A

**解析：** right 表示保留左侧、删除右侧，而对话最新轮次在右侧，需要改为 truncation_side="left"。


### 问题3（多选题）

**题目：** 长对话测试面临？
A. 超窗口截断
B. 早期上下文遗忘
C. KV cache 增大
D. 推理时延增加

**解答：** ABCD

**解析：** 四者都是长序列推理的核心挑战。


### 问题4（多选题）

**题目：** 正确构造多轮 history 需要？
A. system 只出现一次
B. 保留历史 assistant 回答
C. 最新 user 提问放最后
D. 超长时优先截断最旧轮次

**解答：** ABCD

**解析：** system 做全局指令，历史按序保留，超长时优先丢弃旧轮次以保留近期上下文。


### 问题5（判断题）

**题目：** history 越长，模型回答质量一定越高。

**解答：** 错

**解析：** 超过有效上下文或引入噪声后，长 history 反而降低相关性与一致性。


### 问题6（判断题）

**题目：** 将 pad_token_id 设为 eos_token_id 可避免填充导致的生成报错，但可能影响停止行为。

**解答：** 对

**解析：** 该做法能规避缺失 pad token 的异常，但生成时可能把填充位置当结束信号，需要结合停止条件处理。


### 问题7（填空题）

**题目：** 控制输入不超过窗口应设置 max_length 与 ____。

**解答：** truncation=True（以及 truncation_side="left"）


### 问题8（填空题）

**题目：** KV cache 大小随序列长度近似 ____ 增长。

**解答：** 线性


### 问题9（简答题）

**题目：** 为什么超长对话优先保留近期上下文，而不是均等保留所有轮次？

**解答：** 模型注意力对近期信息更敏感，最新 user 问题直接决定回答；早期信息可通过摘要或检索保留，因此窗口受限时应优先丢弃最旧轮次。


### 问题10（简答题）

**题目：** 如何验证模型记住了第 1 轮的细节？

**解答：** 在第 3~4 轮故意引用第 1 轮提到的姓名、地点或事件细节提问，检查回答是否一致；多次随机化问题顺序，统计跨轮一致性得分。


### 问题11（代码设计题）

**题目：** 编写 chat_once(history, user_input)，构造完整 prompt，并返回 (prompt, 更新后的 history)。

**解答：** ```python
def chat_once(history, user_input):
    prompt = "<|system|>\n你是心理咨询助手。\n"
    for h in history:
        prompt += f"<|user|>\n{h[0]}\n<|assistant|>\n{h[1]}\n"
    prompt += f"<|user|>\n{user_input}\n<|assistant|>\n"
    new_history = history + [(user_input, "")]
    return prompt, new_history
```


### 问题12（单选题）

**题目：** 生成结果中出现 <|user|>，正确处理是？
A. 在该标记处截断回答
B. 保留
C. 报错
D. 忽略

**解答：** A

**解析：** user 标记代表模型越界进入下一轮，应截断以免污染对话。


### 问题13（多选题）

**题目：** 长对话评测维度包括？
A. 上下文一致性
B. 追问能力
C. 安全边界
D. 建议相关性

**解答：** ABCD

**解析：** 四者共同衡量多轮场景下的综合表现。


### 问题14（判断题）

**题目：** 多轮评测应固定 seed、生成参数与提示模板，才能公平对比。

**解答：** 对

**解析：** 不固定生成参数会导致同一问题输出不同，无法归因于模型差异。


### 问题15（简答题）

**题目：** 估算 10 轮对话 token 开销并给出降低开销的方案。

**解答：** 每轮 user+assistant 约 200~400 token，10 轮累计 2000~4000+ token，加上 KV cache 显存随长度线性增长；可通过限制轮数、对早期轮次做摘要、滑动窗口截断或检索增强只保留相关片段来降低开销。
